# Samples, the LLN and the CLT: practice notebook

**Causal and Predictive Analytics · Asynchronous module** · Simon Business School · Prof. Tianli Xia

This notebook has all the code from the module's web page, plus a few short exercises (cells marked `# YOUR CODE HERE`). It needs no data files: every example is simulated.

**In Colab:** *File → Save a copy in Drive* first, so your changes are kept. Run cells with **Shift+Enter**.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

## Part A. The LLN, the CLT, and a confidence interval

Same code as on the page: a coin with $p = 0.66$, 5,000 sample means of skewed "customer spend", and a 95% interval from one sample of 400.

In [ ]:
rng = np.random.default_rng(1)

# 1. LLN: running share of heads for a coin with p = 0.66
flips = rng.random(10_000) < 0.66
running = flips.cumsum() / np.arange(1, 10_001)
print("share of heads after 10, 100, 10,000 flips:", running[[9, 99, 9_999]].round(3))

# 2. CLT: 5,000 sample means, each from n = 30 skewed draws (customer spend)
n = 30
means = rng.lognormal(3.4, 0.9, size=(5_000, n)).mean(axis=1)
plt.hist(means, bins=50, color="#2563eb")
plt.title(f"5,000 sample means, n = {n}")
plt.xlabel("sample mean ($)")
plt.show()

# 3. A 95% confidence interval from one sample of 400 customers
x = rng.lognormal(3.4, 0.9, size=400)
se = x.std(ddof=1) / np.sqrt(len(x))
print(f"mean {x.mean():.2f}, SE {se:.2f}, 95% CI [{x.mean() - 1.96*se:.2f}, {x.mean() + 1.96*se:.2f}]")

**Exercise A.** Check what "95% confidence" means. The true mean of this spend population is $\mu = e^{3.4 + 0.9^2/2} \approx 44.93$.
Draw 2,000 samples of $n = 100$, build the interval $\bar X \pm 1.96\, s/\sqrt{n}$ for each, and compute the share of intervals that contain $\mu$. Then try $n = 10$ and $n = 1{,}000$. Why is the coverage below 95% for small $n$?

In [ ]:
mu = np.exp(3.4 + 0.9**2 / 2)
# YOUR CODE HERE

## Part B. Small samples produce the extremes

200 stores, all with the same true conversion rate of 10%. They differ only in how many visitors they had this month (20 to 5,000). Which stores end up at the top and the bottom of the ranking?

In [ ]:
rng = np.random.default_rng(77)
visitors = np.round(np.exp(rng.uniform(np.log(20), np.log(5_000), size=200))).astype(int)
buyers = rng.binomial(visitors, 0.10)          # every store's true rate is 10%
rate = buyers / visitors

order = np.argsort(rate)                         # lowest rate first
print("median visitors, all 200 stores:", int(np.median(visitors)))
print("median visitors, top 10 stores: ", int(np.median(visitors[order[-10:]])))
print("median visitors, bottom 10:     ", int(np.median(visitors[order[:10]])))

plt.scatter(visitors, rate, s=12, color="#94a3b8")
plt.scatter(visitors[order[-10:]], rate[order[-10:]], s=30, color="#16a34a", label="top 10")
plt.scatter(visitors[order[:10]], rate[order[:10]], s=30, color="#dc2626", label="bottom 10")
plt.axhline(0.10, color="black", lw=1)
plt.xscale("log")
plt.xlabel("visitors this month (log scale)")
plt.ylabel("share who bought")
plt.legend()
plt.show()

**Exercise B.** Repeat the month 1,000 times with the same 200 stores (new `buyers` each time). In what share of months is the top-ranked store one of the 50 smallest stores? If store size did not matter, the answer would be 25%.

In [ ]:
# YOUR CODE HERE

## Part C. From an interval to a test

Marketing claims that customers spend $40 on average. Our survey of 400 customers: $\bar X = 45.20$, $s = 50$. Test $H_0: \mu = 40$.

In [ ]:
xbar, s, n, mu0 = 45.20, 50, 400, 40
se = s / np.sqrt(n)
t = (xbar - mu0) / se
p = 2 * stats.norm.cdf(-abs(t))
print(f"SE {se:.2f}, t = {t:.2f}, p = {p:.3f}")
print(f"95% CI [{xbar - 1.96*se:.2f}, {xbar + 1.96*se:.2f}]: is $40 inside? {xbar - 1.96*se < mu0 < xbar + 1.96*se}")

**Exercise C.** Draw one sample of 400 customers from the spend population of Part A (`rng.lognormal(3.4, 0.9, size=400)` with `rng = np.random.default_rng(5)`). Test $H_0: \mu = 40$ twice: by hand as above, and with `stats.ttest_1samp(x, 40)`. Do the two $p$-values agree? Then test $H_0: \mu = 44.93$, the true mean. Would you expect to reject it?

In [ ]:
# YOUR CODE HERE